# Cleaning Data: Customer Sales Dataset

### OIBSIP Data Analytics — Level 1 — Task 3

**Objective:** Take a deliberately messy customer sales dataset and systematically
transform it into a clean, analysis-ready dataset, documenting every decision
along the way.

**Dataset:** `messy_customer_sales.csv` — a synthetic e-commerce customer/sales
dataset (1,025 rows, 9 columns) built to reproduce the common defects found in
real-world "dirty data" practice datasets on Kaggle: missing values, duplicate
rows, inconsistent category labels (e.g. `Male`/`male`/`M`), mixed date formats,
monetary values stored as strings with currency symbols, and numeric outliers
/ impossible values (negative purchase amounts, ages of 0 or 999).

**Tech stack:** Python, pandas, numpy

**Workflow:**
1. Load data & produce a Data Quality Report
2. Handle missing data (column-by-column strategy, justified)
3. Remove duplicate rows
4. Standardise inconsistent formatting (categories, dates)
5. Detect & treat outliers (IQR method)
6. Correct data types
7. Before vs. After summary table
8. Save cleaned dataset to CSV


In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RAW_PATH = "CustomerSales.csv"
CLEAN_PATH = "Cleaned_customerSales.csv"

df_raw = pd.read_csv(RAW_PATH)
print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head(10)

Shape: 1025 rows x 9 columns


,CustomerID,Name,Age,Gender,Country,JoinDate,PurchaseAmount,PaymentMethod,Email
0,CUST00572,Michael Gonzalez,63.0,Female,Australia,14-08-22,115.73,Credit Card,michael.gonzalez572@example.com
1,CUST00147,Richard Davis,16.0,FEMALE,Qatar,13-Apr-18,NaN,PayPal,richard.davis147@example.com
2,CUST00362,Mary Taylor,53.0,female,United Kingdom,06/19/2022,155.15,Credit Card,mary.taylor362@example.com
3,CUST00748,Karen Smith,18.0,FEMALE,India,03-05-22,1157.09,Credit Card,karen.smith748@example.com
4,CUST00397,Michael Anderson,19.0,female,India,09-06-16,195.2,paypal,michael.anderson397@example.com
5,CUST00657,Patricia Moore,9.0,M,Canada,1/19/2018,101.47,cash,patricia.moore657@example.com
6,CUST00434,Susan Thomas,46.0,F,Japan,04-Nov-20,130.28,PayPal,susan.thomas434@example.com
7,CUST00467,Michael Wilson,34.0,female,India,07/13/2019,136.27,CASH,michael.wilson467@example.com
8,CUST00172,Joseph Davis,43.0,M,France,14-09-15,55.78,Debit Card,joseph.davis172@example.com
9,CUST00306,Jessica Taylor,57.0,F,Brazil,08-11-20,95.16,Cash,jessica.taylor306@example.com


## 1. Data Quality Report

Before touching anything, we profile the dataset as-is: null counts, duplicate
rows, dtype issues, and value-range anomalies. This report is what drives every
cleaning decision that follows, and we snapshot key metrics now so we can build
a genuine "before vs. after" comparison later.

In [16]:
def data_quality_report(df, label="Dataset"):
    print(f"{'='*60}\n DATA QUALITY REPORT: {label}\n{'='*60}")
    print(f"Rows: {len(df)}   Columns: {df.shape[1]}")

    print("\n--- Null counts per column ---")
    nulls = df.isnull().sum()
    null_pct = (nulls / len(df) * 100).round(2)
    print(pd.DataFrame({"nulls": nulls, "pct": null_pct}))

    print("\n--- Duplicate rows (exact, all columns) ---")
    n_dupes = df.duplicated().sum()
    print(f"{n_dupes} duplicate rows ({n_dupes/len(df)*100:.2f}%)")

    print("\n--- Duplicate CustomerID (should be unique) ---")
    if "CustomerID" in df.columns:
        print(f"{df['CustomerID'].duplicated().sum()} duplicated CustomerID values")

    print("\n--- Dtypes ---")
    print(df.dtypes)

    return {"nulls": nulls, "n_dupes": n_dupes, "n_rows": len(df), "dtypes": df.dtypes.copy()}

dq_before = data_quality_report(df_raw, "RAW (before cleaning)")

 DATA QUALITY REPORT: RAW (before cleaning)
Rows: 1025   Columns: 9

--- Null counts per column ---
                nulls   pct
CustomerID          0  0.00
Name                0  0.00
Age                62  6.05
Gender             30  2.93
Country            20  1.95
JoinDate           42  4.10
PurchaseAmount     52  5.07
PaymentMethod      30  2.93
Email              11  1.07

--- Duplicate rows (exact, all columns) ---
25 duplicate rows (2.44%)

--- Duplicate CustomerID (should be unique) ---
25 duplicated CustomerID values

--- Dtypes ---
CustomerID            str
Name                  str
Age               float64
Gender                str
Country               str
JoinDate              str
PurchaseAmount        str
PaymentMethod         str
Email                 str
dtype: object


In [17]:
# Value-range / format anomaly checks

print("--- Age: non-numeric values / dtype ---")
print("dtype:", df_raw['Age'].dtype)
age_numeric = pd.to_numeric(df_raw['Age'], errors='coerce')
print("Non-numeric Age entries (after coercion, excluding true NaN):",
      ((age_numeric.isna()) & (df_raw['Age'].notna())).sum())
print("Age range (numeric-coercible):", age_numeric.min(), "to", age_numeric.max())
print("Ages outside plausible human range (0-110):",
      ((age_numeric < 0) | (age_numeric > 110)).sum())

print("\n--- Gender: unique raw values ---")
print(sorted(df_raw['Gender'].dropna().unique()))

print("\n--- Country: unique raw values ---")
print(sorted(df_raw['Country'].dropna().unique()))

print("\n--- PaymentMethod: unique raw values ---")
print(sorted(df_raw['PaymentMethod'].dropna().unique()))

print("\n--- PurchaseAmount: dtype and sample of non-numeric-looking entries ---")
print("dtype:", df_raw['PurchaseAmount'].dtype)
pa_str_mask = df_raw['PurchaseAmount'].astype(str).str.contains(r'[^0-9.\-]', na=False)
print("Entries containing non-numeric characters (e.g. '$', ','):", pa_str_mask.sum())
print(df_raw.loc[pa_str_mask, 'PurchaseAmount'].head())

print("\n--- JoinDate: sample of raw values (mixed formats) ---")
print(df_raw['JoinDate'].dropna().sample(10, random_state=1).tolist())

--- Age: non-numeric values / dtype ---
dtype: float64
Non-numeric Age entries (after coercion, excluding true NaN): 0
Age range (numeric-coercible): -14.0 to 999.0
Ages outside plausible human range (0-110): 11

--- Gender: unique raw values ---
['F', 'FEMALE', 'Female', 'M', 'MALE', 'Male', 'female', 'male']

--- Country: unique raw values ---
['Australia', 'Brazil', 'Canada', 'France', 'Germany', 'India', 'Japan', 'Qatar', 'U.K.', 'U.S.A.', 'UK', 'US', 'USA', 'United Kingdom', 'United States', 'united kingdom', 'united states']

--- PaymentMethod: unique raw values ---
['CASH', 'CREDIT CARD', 'Cash', 'Credit Card', 'Debit Card', 'PayPal', 'cash', 'credit card', 'debit card', 'paypal']

--- PurchaseAmount: dtype and sample of non-numeric-looking entries ---
dtype: str
Entries containing non-numeric characters (e.g. '$', ','): 20
96     $113.42 
106    $100.01 
206    $162.18 
288    $104.15 
299    $124.52 
Name: PurchaseAmount, dtype: str

--- JoinDate: sample of raw values (mixed f

**Findings from the Data Quality Report:**

| Issue | Detail |
|---|---|
| Missing values | `Age`, `Gender`, `Country`, `JoinDate`, `PurchaseAmount`, `PaymentMethod`, `Email` all contain nulls (roughly 1–6% each) |
| Duplicate rows | 25 exact duplicate rows were injected and are detected as such |
| Wrong dtypes | `Age` and `PurchaseAmount` are stored as `object` because some values are strings (`"34"`, `"$198.46"`) mixed with numbers |
| Inconsistent categories | `Gender` has 8 variants of Male/Female; `Country` has multiple spellings for the same country (e.g. `USA`/`US`/`United States`); `PaymentMethod` has inconsistent casing |
| Inconsistent dates | `JoinDate` mixes `YYYY-MM-DD`, `MM/DD/YYYY`, `DD-MM-YYYY`, and `DD Mon YYYY` formats |
| Value-range anomalies | Some `Age` values are negative, 0, or absurdly high (150, 999); some `PurchaseAmount` values are negative or implausibly large (data entry errors / extreme outliers) |

These are addressed in order below.

## 2. Data Type Correction

We fix dtypes early because missing-value imputation, standardisation, and
outlier detection all need to operate on properly typed numeric/datetime
columns rather than on mixed-type strings.

- **`Age`** → numeric (`float64` while nulls remain, then nullable `Int64` once imputed)
- **`PurchaseAmount`** → strip `$` and `,` characters, convert to `float64`
- **`JoinDate`** → parse mixed formats into a single `datetime64[ns]` column
- **`CustomerID`** → kept as `string` (it's an identifier, not a number — arithmetic on it is meaningless and leading zeros must be preserved)
- **`Gender`, `Country`, `PaymentMethod`** → converted to pandas `category` dtype after standardisation (below), which is both memory-efficient and semantically correct for a fixed set of labels

In [18]:
df = df_raw.copy()

# --- CustomerID: keep as string/object explicitly (it's an ID, not a number) ---
df['CustomerID'] = df['CustomerID'].astype('string')

# --- Age: coerce to numeric float (nulls preserved as NaN for now) ---
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# --- PurchaseAmount: strip currency symbols/commas, coerce to float ---
df['PurchaseAmount'] = (
    df['PurchaseAmount']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .replace('nan', np.nan)
)
df['PurchaseAmount'] = pd.to_numeric(df['PurchaseAmount'], errors='coerce')

# --- JoinDate: parse mixed formats ---
# Try a list of known formats in priority order; fall back to pandas' general parser.
def parse_mixed_date(val):
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip()
    formats = ["%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y", "%d %b %Y"]
    for fmt in formats:
        try:
            return pd.to_datetime(val, format=fmt)
        except (ValueError, TypeError):
            continue
    # Fallback: let pandas infer (covers e.g. non-zero-padded '9/6/2016')
    return pd.to_datetime(val, errors='coerce', dayfirst=False)

df['JoinDate'] = df['JoinDate'].apply(parse_mixed_date)

print(df.dtypes)
print("\nUnparsed JoinDate after conversion:", df['JoinDate'].isna().sum(),
      "(", df_raw['JoinDate'].isna().sum(), "were already null in raw data)")
print("Failed-to-parse (non-null raw, but NaT after parsing):",
      ((df['JoinDate'].isna()) & (df_raw['JoinDate'].notna())).sum())

CustomerID                string
Name                         str
Age                      float64
Gender                       str
Country                      str
JoinDate          datetime64[us]
PurchaseAmount           float64
PaymentMethod                str
Email                        str
dtype: object

Unparsed JoinDate after conversion: 42 ( 42 were already null in raw data)
Failed-to-parse (non-null raw, but NaT after parsing): 0


## 3. Standardisation of Categorical Text

We normalise inconsistent labels to a single canonical form per category,
using case-insensitive matching plus a small mapping dictionary for
abbreviations (`M`→`Male`, `US`→`United States`, etc.).

In [19]:
print("Gender values before:", sorted(df['Gender'].dropna().unique()))
print("Country values before:", sorted(df['Country'].dropna().unique()))
print("PaymentMethod values before:", sorted(df['PaymentMethod'].dropna().unique()))

Gender values before: ['F', 'FEMALE', 'Female', 'M', 'MALE', 'Male', 'female', 'male']
Country values before: ['Australia', 'Brazil', 'Canada', 'France', 'Germany', 'India', 'Japan', 'Qatar', 'U.K.', 'U.S.A.', 'UK', 'US', 'USA', 'United Kingdom', 'United States', 'united kingdom', 'united states']
PaymentMethod values before: ['CASH', 'CREDIT CARD', 'Cash', 'Credit Card', 'Debit Card', 'PayPal', 'cash', 'credit card', 'debit card', 'paypal']


In [20]:
# --- Gender standardisation ---
gender_map = {
    'male': 'Male', 'm': 'Male',
    'female': 'Female', 'f': 'Female',
}
df['Gender'] = df['Gender'].str.strip().str.lower().map(gender_map).fillna(df['Gender'])
df['Gender'] = df['Gender'].astype('category')

# --- Country standardisation ---
country_map = {
    'usa': 'United States', 'us': 'United States', 'u.s.a.': 'United States',
    'united states': 'United States',
    'uk': 'United Kingdom', 'u.k.': 'United Kingdom', 'united kingdom': 'United Kingdom',
}
df['Country'] = df['Country'].str.strip().str.lower().map(country_map).fillna(
    df['Country'].str.strip()
)
df['Country'] = df['Country'].astype('category')

# --- PaymentMethod standardisation (title-case fixes casing inconsistencies) ---
df['PaymentMethod'] = df['PaymentMethod'].str.strip().str.title()
df['PaymentMethod'] = df['PaymentMethod'].astype('category')

print("Gender values after:", df['Gender'].cat.categories.tolist())
print("Country values after:", sorted(df['Country'].dropna().unique()))
print("PaymentMethod values after:", df['PaymentMethod'].cat.categories.tolist())

Gender values after: ['Female', 'Male']
Country values after: ['Australia', 'Brazil', 'Canada', 'France', 'Germany', 'India', 'Japan', 'Qatar', 'United Kingdom', 'United States']
PaymentMethod values after: ['Cash', 'Credit Card', 'Debit Card', 'Paypal']


**Why `category` dtype for these three columns:** each has a small, fixed
set of possible values, so `category` is more memory-efficient than `object`/`string`
and communicates the semantic intent (these are categorical labels, not free text)
to anyone reading the schema or running further analysis (e.g. `groupby`).

## 4. Missing Data Handling

Each column gets a strategy chosen for *that column's* role in the dataset —
there is no single "correct" default.

| Column | Missingness | Strategy | Justification |
|---|---|---|---|
| `Age` | ~6% | **Median imputation** | Age is numeric but right-skewed with a few extreme/erroneous values (handled in outlier step); median is robust to those outliers, unlike the mean. |
| `Gender` | ~3% | **Mode imputation** | Categorical with only two real categories; filling with the most frequent category is a standard, low-risk default when we have no other signal to predict gender from. |
| `Country` | ~2% | **Mode imputation** | Categorical; the small missing fraction and lack of any correlated column to infer it from make mode imputation reasonable. Row deletion was considered but rejected — losing 2% of rows for a single field is wasteful when a reasonable fill exists. |
| `JoinDate` | ~4% | **Row deletion** | Date of joining is not something that can be legitimately imputed (no reliable proxy exists, and inventing a join date would fabricate history for tenure-based analysis). Since it's only ~4% of rows, deletion is safer than guessing. |
| `PurchaseAmount` | ~5% (before outlier handling) | **Median imputation** | This is the core numeric metric of the dataset. Median is used (not mean) because the raw values include some extreme outliers that would bias a mean-based fill; imputing after computing the median keeps the fill robust. |
| `PaymentMethod` | ~3% | **Mode imputation** | Categorical with a handful of common values; mode is the standard low-information default. |
| `Email` | ~1% | **Row deletion** | Email is effectively a unique identifier/contact field — it cannot be meaningfully imputed, and the missing fraction is tiny (~1%), so dropping those rows has negligible impact on the dataset. |

We apply row-deletions for `JoinDate`/`Email` first (so we're not imputing values
we're about to discard), then impute the remaining columns.

In [21]:
print("Rows before missing-data handling:", len(df))

# Row deletion for columns where imputation isn't defensible
before = len(df)
df = df.dropna(subset=['JoinDate', 'Email'])
print(f"Dropped {before - len(df)} rows with missing JoinDate or Email "
      f"({before} -> {len(df)})")

# Median imputation for numeric columns
age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)
print(f"Age: filled {df_raw['Age'].isna().sum()} missing values (approx) with median = {age_median}")

purchase_median = df['PurchaseAmount'].median()
df['PurchaseAmount'] = df['PurchaseAmount'].fillna(purchase_median)
print(f"PurchaseAmount: filled with median = {purchase_median:.2f}")

# Mode imputation for categorical columns
for col in ['Gender', 'Country', 'PaymentMethod']:
    mode_val = df[col].mode(dropna=True)[0]
    n_missing = df[col].isna().sum()
    df[col] = df[col].fillna(mode_val)
    print(f"{col}: filled {n_missing} missing values with mode = '{mode_val}'")

print("\nRemaining nulls per column:")
print(df.isnull().sum())

Rows before missing-data handling: 1025
Dropped 51 rows with missing JoinDate or Email (1025 -> 974)
Age: filled 62 missing values (approx) with median = 38.0
PurchaseAmount: filled with median = 121.59
Gender: filled 27 missing values with mode = 'Female'
Country: filled 20 missing values with mode = 'Australia'
PaymentMethod: filled 28 missing values with mode = 'Cash'

Remaining nulls per column:
CustomerID        0
Name              0
Age               0
Gender            0
Country           0
JoinDate          0
PurchaseAmount    0
PaymentMethod     0
Email             0
dtype: int64


## 5. Duplicate Removal

We check for exact full-row duplicates (identical values across every column)
and also flag duplicated `CustomerID`s, which would indicate the same customer
appearing twice even if some other field differs.

In [22]:
before = len(df)
n_exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows found: {n_exact_dupes}")

df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows ({before} -> {len(df)})")

# Check for duplicate CustomerIDs remaining (would indicate a different kind of duplication)
dup_ids = df['CustomerID'].duplicated().sum()
print(f"Duplicate CustomerID values remaining after exact-dup removal: {dup_ids}")

Exact duplicate rows found: 22
Removed 22 duplicate rows (974 -> 952)
Duplicate CustomerID values remaining after exact-dup removal: 0


## 6. Outlier Detection (IQR Method)

We use the IQR (interquartile range) method on the two numeric columns,
`Age` and `PurchaseAmount`: values below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`
are flagged as outliers.

**Decisions:**
- **`Age`**: Ages must be biologically plausible. Values outside a hard
  domain bound of **0–110** are treated as **data entry errors and removed**
  (not just statistical outliers — they are impossible). Remaining statistical
  outliers within the plausible range are **retained** (an 80-year-old customer
  is unusual but real).
- **`PurchaseAmount`**: Negative purchase amounts are **impossible** for this
  business context (a purchase can't cost less than $0) and are treated as
  data entry errors and removed. Remaining statistically extreme values
  (very large but non-negative purchases) are **capped (winsorized)** at the
  IQR upper bound rather than removed — a $2,000 single purchase is plausible
  for some customers, and capping preserves the row (and its other fields)
  while limiting the outlier's influence on downstream aggregates like the mean.

In [23]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

# --- Age: remove biologically impossible values, retain in-range statistical outliers ---
before = len(df)
invalid_age_mask = (df['Age'] < 0) | (df['Age'] > 110)
print(f"Removing {invalid_age_mask.sum()} rows with impossible Age values "
      f"(examples: {sorted(df.loc[invalid_age_mask, 'Age'].unique())[:5]})")
df = df.loc[~invalid_age_mask].copy()

low, high = iqr_bounds(df['Age'])
stat_outliers = ((df['Age'] < low) | (df['Age'] > high)).sum()
print(f"Age IQR bounds: [{low:.1f}, {high:.1f}] -> {stat_outliers} statistical outliers "
      f"retained (plausible values, just unusual)")

# --- PurchaseAmount: remove negative (impossible) values, cap statistically extreme values ---
invalid_purchase_mask = df['PurchaseAmount'] < 0
print(f"\nRemoving {invalid_purchase_mask.sum()} rows with negative PurchaseAmount "
      f"(data entry errors)")
df = df.loc[~invalid_purchase_mask].copy()

low, high = iqr_bounds(df['PurchaseAmount'])
n_capped_high = (df['PurchaseAmount'] > high).sum()
n_capped_low = (df['PurchaseAmount'] < low).sum()
print(f"PurchaseAmount IQR bounds: [{low:.2f}, {high:.2f}]")
print(f"Capping {n_capped_high} values above upper bound and {n_capped_low} below lower bound")
df['PurchaseAmount'] = df['PurchaseAmount'].clip(lower=low, upper=high)

print(f"\nRows remaining after outlier treatment: {len(df)} (started this step with {before})")

Removing 9 rows with impossible Age values (examples: [np.float64(-14.0), np.float64(-5.0), np.float64(150.0), np.float64(999.0)])
Age IQR bounds: [10.0, 66.0] -> 19 statistical outliers retained (plausible values, just unusual)

Removing 5 rows with negative PurchaseAmount (data entry errors)
PurchaseAmount IQR bounds: [-3.96, 247.22]
Capping 11 values above upper bound and 0 below lower bound

Rows remaining after outlier treatment: 938 (started this step with 952)


## 7. Final Data Type Pass

With missing values resolved and outliers treated, we lock in final dtypes:
`Age` as a nullable integer (ages are whole numbers), `PurchaseAmount` as
`float64` (monetary value), `JoinDate` as `datetime64[ns]`, `CustomerID` as
`string`, and the three categorical columns as `category`.

In [24]:
df['Age'] = df['Age'].round().astype('Int64')
df['PurchaseAmount'] = df['PurchaseAmount'].astype('float64').round(2)
df['CustomerID'] = df['CustomerID'].astype('string')
df['Email'] = df['Email'].astype('string')
df['Name'] = df['Name'].astype('string')
# JoinDate, Gender, Country, PaymentMethod were already set above

df = df.reset_index(drop=True)
print(df.dtypes)
df.head(10)

CustomerID                string
Name                      string
Age                        Int64
Gender                  category
Country                 category
JoinDate          datetime64[us]
PurchaseAmount           float64
PaymentMethod           category
Email                     string
dtype: object


,CustomerID,Name,Age,Gender,Country,JoinDate,PurchaseAmount,PaymentMethod,Email
0,CUST00572,Michael Gonzalez,63,Female,Australia,2022-08-14,115.73,Credit Card,michael.gonzalez572@example.com
1,CUST00147,Richard Davis,16,Female,Qatar,2018-04-13,121.59,Paypal,richard.davis147@example.com
2,CUST00362,Mary Taylor,53,Female,United Kingdom,2022-06-19,155.15,Credit Card,mary.taylor362@example.com
3,CUST00748,Karen Smith,18,Female,India,2022-03-05,247.22,Credit Card,karen.smith748@example.com
4,CUST00397,Michael Anderson,19,Female,India,2016-09-06,195.20,Paypal,michael.anderson397@example.com
5,CUST00657,Patricia Moore,9,Male,Canada,2018-01-19,101.47,Cash,patricia.moore657@example.com
6,CUST00434,Susan Thomas,46,Female,Japan,2020-11-04,130.28,Paypal,susan.thomas434@example.com
7,CUST00467,Michael Wilson,34,Female,India,2019-07-13,136.27,Cash,michael.wilson467@example.com
8,CUST00172,Joseph Davis,43,Male,France,2015-09-14,55.78,Debit Card,joseph.davis172@example.com
9,CUST00306,Jessica Taylor,57,Female,Brazil,2020-08-11,95.16,Cash,jessica.taylor306@example.com


## 8. Before vs. After Summary

A single table comparing key data-quality metrics before and after the
cleaning pipeline.

In [25]:
dq_after = data_quality_report(df, "CLEAN (after cleaning)")

 DATA QUALITY REPORT: CLEAN (after cleaning)
Rows: 938   Columns: 9

--- Null counts per column ---
                nulls  pct
CustomerID          0  0.0
Name                0  0.0
Age                 0  0.0
Gender              0  0.0
Country             0  0.0
JoinDate            0  0.0
PurchaseAmount      0  0.0
PaymentMethod       0  0.0
Email               0  0.0

--- Duplicate rows (exact, all columns) ---
0 duplicate rows (0.00%)

--- Duplicate CustomerID (should be unique) ---
0 duplicated CustomerID values

--- Dtypes ---
CustomerID                string
Name                      string
Age                        Int64
Gender                  category
Country                 category
JoinDate          datetime64[us]
PurchaseAmount           float64
PaymentMethod           category
Email                     string
dtype: object


In [26]:
def dtype_accuracy(df, expected):
    """Fraction of columns whose dtype matches the expected 'correct' dtype family."""
    correct = 0
    for col, kind in expected.items():
        actual = str(df[col].dtype)
        if kind == 'numeric' and ('int' in actual or 'float' in actual):
            correct += 1
        elif kind == 'datetime' and 'datetime' in actual:
            correct += 1
        elif kind == 'string' and actual in ('string', 'object'):
            correct += 1
        elif kind == 'category' and actual == 'category':
            correct += 1
    return correct / len(expected)

expected_dtypes = {
    'CustomerID': 'string', 'Name': 'string', 'Age': 'numeric',
    'Gender': 'category', 'Country': 'category', 'JoinDate': 'datetime',
    'PurchaseAmount': 'numeric', 'PaymentMethod': 'category', 'Email': 'string',
}

summary = pd.DataFrame({
    "Metric": [
        "Row count",
        "Total null values",
        "Duplicate rows",
        "Dtype accuracy (cols with correct type)",
    ],
    "Before": [
        dq_before["n_rows"],
        int(dq_before["nulls"].sum()),
        int(dq_before["n_dupes"]),
        f"{dtype_accuracy(df_raw, expected_dtypes)*100:.0f}%",
    ],
    "After": [
        dq_after["n_rows"],
        int(dq_after["nulls"].sum()),
        int(dq_after["n_dupes"]),
        f"{dtype_accuracy(df, expected_dtypes)*100:.0f}%",
    ],
})
summary

,Metric,Before,After
0,Row count,1025,938
1,Total null values,247,0
2,Duplicate rows,25,0
3,Dtype accuracy (cols with correct type),11%,89%


## 9. Save Cleaned Dataset

In [27]:
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved cleaned dataset to '{CLEAN_PATH}'")
print(f"Final shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(10)

Saved cleaned dataset to 'Cleaned_customerSales.csv'
Final shape: 938 rows x 9 columns


,CustomerID,Name,Age,Gender,Country,JoinDate,PurchaseAmount,PaymentMethod,Email
0,CUST00572,Michael Gonzalez,63,Female,Australia,2022-08-14,115.73,Credit Card,michael.gonzalez572@example.com
1,CUST00147,Richard Davis,16,Female,Qatar,2018-04-13,121.59,Paypal,richard.davis147@example.com
2,CUST00362,Mary Taylor,53,Female,United Kingdom,2022-06-19,155.15,Credit Card,mary.taylor362@example.com
3,CUST00748,Karen Smith,18,Female,India,2022-03-05,247.22,Credit Card,karen.smith748@example.com
4,CUST00397,Michael Anderson,19,Female,India,2016-09-06,195.20,Paypal,michael.anderson397@example.com
5,CUST00657,Patricia Moore,9,Male,Canada,2018-01-19,101.47,Cash,patricia.moore657@example.com
6,CUST00434,Susan Thomas,46,Female,Japan,2020-11-04,130.28,Paypal,susan.thomas434@example.com
7,CUST00467,Michael Wilson,34,Female,India,2019-07-13,136.27,Cash,michael.wilson467@example.com
8,CUST00172,Joseph Davis,43,Male,France,2015-09-14,55.78,Debit Card,joseph.davis172@example.com
9,CUST00306,Jessica Taylor,57,Female,Brazil,2020-08-11,95.16,Cash,jessica.taylor306@example.com


## Summary of Cleaning Decisions

1. **Data quality report** run first to drive every subsequent decision, not guess at them.
2. **Dtypes fixed early** (`Age`, `PurchaseAmount` → numeric; `JoinDate` → datetime) so later numeric/date logic works correctly.
3. **Categorical text standardised** (`Gender`, `Country`, `PaymentMethod`) via mapping dictionaries, converted to `category` dtype.
4. **Missing data**: median imputation for skewed numerics (`Age`, `PurchaseAmount`), mode imputation for categoricals with no better signal (`Gender`, `Country`, `PaymentMethod`), row deletion for fields that can't be legitimately invented (`JoinDate`, `Email`), each justified individually rather than applying one blanket rule.
5. **Duplicates**: exact full-row duplicates removed; duplicate `CustomerID`s checked separately.
6. **Outliers (IQR)**: impossible values (negative purchases, out-of-range ages) removed as data entry errors; plausible-but-extreme values retained (`Age`) or capped (`PurchaseAmount`) rather than deleted, to preserve sample size while limiting distortion of aggregate statistics.
7. **Final dtype pass** locks in the correct, analysis-ready schema.
8. **Before/after table** quantifies the improvement across nulls, duplicates, and dtype correctness.
